In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def zero_one(pred, y): return np.mean(np.asarray(pred) != np.asarray(y))
print("setup ok")

In [ ]:
X=np.array([1,2,3,4,5]); y=np.array([0,0,1,1,1])
def risk(t): return zero_one((X>t).astype(int), y)
for t in [1.5,2.5,3.5]: print(f"t={t}: R_S={risk(t):.2f}")
assert abs(risk(1.5)-0.2)<1e-9 and risk(2.5)==0.0 and abs(risk(3.5)-0.2)<1e-9

In [ ]:
# ERM scans the class and returns the minimizer
ts=np.linspace(0,6,61); erm_t=ts[np.argmin([risk(t) for t in ts])]
print("ERM threshold ~", round(float(erm_t),2)); assert 2.0<=erm_t<=3.0

In [ ]:
ys=np.array([2.,4.,6.])
for c in [3,4,5]: print(f"c={c}: MSE={np.mean((c-ys)**2):.3f}")
assert abs(np.mean((4-ys)**2)-2.667)<1e-3 and abs(np.mean(ys)-4)<1e-9

In [ ]:
rng=np.random.default_rng(1); n=40
xs=np.sort(rng.uniform(0,1,n)); ys=(xs>0.5).astype(int)
flip=rng.random(n)<0.1; ys=ys^flip           # 10% label noise
def knn_train_err(k):
    e=0
    for i in range(n):
        idx=np.argsort(np.abs(xs-xs[i]))[:k]
        e+=(round(ys[idx].mean())!=ys[i])
    return e/n
for k in [1,3,7,15]: print(f"k={k:2d}-NN: train err={knn_train_err(k):.3f}")
assert knn_train_err(1)==0.0   # 1-NN memorizes -> zero training error

In [ ]:
def knn_true_err(k):
    grid=np.linspace(0,1,400); e=0
    for gx in grid:
        idx=np.argsort(np.abs(xs-gx))[:k]
        e+=(round(ys[idx].mean())!=int(gx>0.5))
    return e/len(grid)
ks=range(1,25)
train=[knn_train_err(k) for k in ks]; tru=[knn_true_err(k) for k in ks]
plt.figure(figsize=(7,4))
plt.plot(list(ks),train,'-o',ms=3,label='training error')
plt.plot(list(ks),tru,'-s',ms=3,label='true error (approx)')
plt.axhline(0.1,ls='--',c='gray',label='Bayes floor (10% noise)')
plt.xlabel('neighborhood size k  (small k = complex)'); plt.ylabel('error'); plt.legend()
plt.title('ERM with 1-NN: zero training error, high true error')
assert train[0]==0.0 and tru[0] > train[0]   # memorization gap